# Room Booking Chatbot — Technologies & Code Walkthrough

Promtior technical challenge (AI Engineer). This notebook explains each technology used in the solution and shows runnable code, from the deterministic core to the tool-calling agent.

## 1. Stack

| Component | Technology | Role |
|---|---|---|
| LLM | **Groq** (Llama 3.3 70B, free API) | language understanding + tool choice |
| Agent framework | **LangGraph** (`create_react_agent`) | reasoning loop: LLM <-> tools |
| Tools | **LangChain `@tool`** | typed, documented functions the model can call |
| Domain core | Pure Python + **sqlite3** | all business rules, deterministic & tested |
| API | **FastAPI** + JWT | REST access + `/chat` endpoint |
| UI | **Streamlit** | conversational interface with login |
| Deploy | Docker | Hugging Face Spaces / Railway |

## 2. The deterministic core (no LLM involved)

Every rule from the challenge lives in `app/booking_core.py`: 30-minute alignment, capacity per room, contiguous slots up to 3h, no overlaps (end-exclusive), office hours, no past bookings, ownership on cancellation. Example:

In [ ]:
import sys, os, tempfile
sys.path.insert(0, os.path.abspath('..'))
from datetime import date, timedelta
from app.booking_core import BookingSystem, BookingError

fd, path = tempfile.mkstemp(suffix='.db'); os.close(fd)
bs = BookingSystem(db_path=path)
tomorrow = (date.today() + timedelta(days=1)).strftime('%Y-%m-%d')

b = bs.create_booking('C', tomorrow, '10:00', '11:30', 'Sprint planning', 8, 'User1')
print('Booked:', b)

# The challenge's own example: 10:00-11:30 blocks any earlier start
try:
    bs.create_booking('C', tomorrow, '10:30', '11:00', 'Nope', 2, 'User2')
except BookingError as e:
    print('Rejected ->', e)

# ...but a booking starting exactly at 11:30 is fine (end-exclusive rule)
print('Back-to-back ok:', bs.create_booking('C', tomorrow, '11:30', '12:00', 'Demo', 3, 'User2').id)

## 3. Why tool-calling?

A plain LLM **hallucinates**: asked to 'book room C at 10', it might invent a success message without writing anything. With tool-calling the model only *proposes* a call (`create_booking(room='C', date=..., start='10:00', ...)`); the tool executes it against the real system and returns the true result (or the validation error). The LLM is the flexible front, the code is the reliable back.

## 4. Tools with LangChain

The `@tool` decorator turns a function into a tool the model can see — its name, docstring and type hints become the model's 'instruction manual'. Note how each tool catches `BookingError` and returns the message: errors flow back to the model so it can explain or retry instead of crashing.

In [ ]:
from langchain_core.tools import tool
from app.booking_core import BookingError

@tool
def create_booking(room: str, date: str, start: str, end: str,
                   title: str, attendees: int) -> str:
    """Book a meeting room. Times must be :00 or :30 aligned; max 3 hours."""
    try:
        b = bs.create_booking(room, date, start, end, title, attendees, 'User1')
        return f'Booking #{b.id} confirmed: room {b.room}, {b.date} {b.start}-{b.end}.'
    except BookingError as e:
        return f'Booking failed: {e}'

print(create_booking.name)
print(create_booking.description[:120], '...')
print(create_booking.invoke({'room': 'C', 'date': tomorrow, 'start': '14:00',
                             'end': '15:30', 'title': 'Interview', 'attendees': 9}))

## 5. The agent: LangGraph + Groq

`create_react_agent` runs the classic ReAct loop: the LLM thinks -> calls a tool -> reads the result -> thinks again ... until it produces a final answer. Groq hosts Llama 3.3 70B with a **free API** (no credit card), fully compatible with LangChain via `langchain-groq`.

⚠️ Requires `GROQ_API_KEY` in the environment (create one at https://console.groq.com).

In [ ]:
import os
from langchain_groq import ChatGroq
from langgraph.prebuilt import create_react_agent
from app.tools import build_tools

assert os.getenv('GROQ_API_KEY'), 'set GROQ_API_KEY first'

llm = ChatGroq(model='llama-3.3-70b-versatile', temperature=0)
tools = build_tools(bs, 'User1')
prompt = ('You are the Cubo Itaú room-booking assistant. '
          'Ask for missing details before booking.')
agent = create_react_agent(llm, tools, prompt=prompt)

for question in ['Which rooms are free tomorrow 10:00-11:00 for 6 people?',
                 'Book room B tomorrow 15:00-15:30 for 3 people, title 1:1']:
    result = agent.invoke({'messages': [('user', question)]})
    print('Q:', question)
    print('A:', result['messages'][-1].content, '\n')

## 6. Conversation memory & the UI

Streamlit keeps `st.session_state.messages` and passes the full history to the agent on every turn, so follow-ups like *'cancel that last one'* work. The UI also handles login (User1/User2, challenge password) before the agent is built, binding every booking to the right user.

```bash
streamlit run ui/streamlit_app.py
```

## 7. REST API (same system, programmatic access)

```bash
uvicorn app.main:app --reload   # interactive docs at /docs
```

```python
import httpx
token = httpx.post('http://localhost:8000/auth/login',
                   json={'username': 'User1', 'password': 'TechnicalChallengePromtior'}).json()['access_token']
headers = {'Authorization': f'Bearer {token}'}
httpx.post('http://localhost:8000/bookings', headers=headers,
           json={'room': 'E', 'date': tomorrow, 'start': '09:00', 'end': '10:00',
                 'title': 'All-hands', 'attendees': 18})
```

## 8. Deployment

Single Dockerfile; runs Streamlit by default, or `uvicorn app.main:app` for the API. Deployed free on **Hugging Face Spaces** (public repo = free streamlit space). Ollama-in-cloud was discarded: 4–12 GB RAM is not viable on free tiers.

**Secrets:** `GROQ_API_KEY` (and `JWT_SECRET`) are set via the platform's environment-variable settings, never committed to the repo.